In [46]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# Ignoro i warning 
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Percorso da cui prendere il file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Serve per far paritre il debug
DEBUG = False

# Carico il csv e stampo la shape

In [47]:
df_duke = pd.read_csv(FILE_PATH / "duke_lesions.csv")

print("DUKE shape: ", df_duke.shape)

DUKE shape:  (291, 109)


# Target

In [48]:
marker = ["PR", "ER"]

# Bilancio il dataset globalmente

In [49]:
"""
    Crea una coorte bilanciata selezionando un numero uguale di campioni 
    per ogni classe (Undersampling). Utilizza solo pazienti reali.
"""

def get_balanced_cohort(df, target, features, random_state=42):
    df_0 = df[df[target] == 0]
    df_1 = df[df[target] == 1]
    
    n_min = min(len(df_0), len(df_1))
    
    df_bal = pd.concat([
        df_0.sample(n=n_min, random_state=random_state),
        df_1.sample(n=n_min, random_state=random_state)
    ]).sample(frac=1, random_state=random_state)
    
    X = df_bal[features]
    y = df_bal[target]
    return X, y

# Controllo il numero delle classi

In [50]:
for m in marker:
    vc = df_duke[m].value_counts(dropna=False)

    print(f"\nDistribuzione {m} – DUKE")
    print("-" * 30)
    print("Negativi:", vc.get(0, 0))
    print("Positivi:", vc.get(1, 0))



Distribuzione PR – DUKE
------------------------------
Negativi: 157
Positivi: 134

Distribuzione ER – DUKE
------------------------------
Negativi: 123
Positivi: 168


# Stratified Cross-Validation

In [51]:
FEATURES = [c for c in df_duke.columns if c.startswith("original_")]

# Training

In [52]:
for target in marker:
    print(f"\n" + "="*80)
    print(f" ANALISI CON COORTE BILANCIATA - TARGET: {target}")
    print(f"="*80)
    
    X, y = get_balanced_cohort(df_duke, target, FEATURES)


    # nei fold 4 e 5 lo socre è diverso perchè facendo la divisione dei fold ho il riporto.
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, bal_acc_scores, f1_scores, auc_scores = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = XGBClassifier(
            random_state=42,
            n_jobs=-1,
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_alpha=0,
            reg_lambda=1
        )
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] # Necessario per la ROC-AUC
        
        # Calcolo delle 4 metriche
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        # Salvataggio nelle liste
        acc_scores.append(acc)
        bal_acc_scores.append(bal_acc)
        f1_scores.append(f1)
        auc_scores.append(auc)

        print(f"\n--- Fold {fold} ---")
        print(f"Supporto Test Set (Coorte Reale): {dict(y_test.value_counts())}")
        print(f"Accuracy: {acc:.4f} | Balanced Accuracy: {bal_acc:.4f}")
        print(classification_report(y_test, y_pred))

    # --- RISULTATI FINALI ADATTATI ---
    print(f"\n" + "-"*40)
    print(f" RISULTATI MEDI FINALI PER {target}")
    print(f" Accuracy Media : {np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}")
    print(f" Balanced Media : {np.mean(bal_acc_scores):.3f} ± {np.std(bal_acc_scores):.3f}")
    print(f" F1-score Medio : {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
    print(f" ROC-AUC Media  : {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")
    print("-"*40)


 ANALISI CON COORTE BILANCIATA - TARGET: PR

--- Fold 1 ---
Supporto Test Set (Coorte Reale): {0: np.int64(27), 1: np.int64(27)}
Accuracy: 0.5556 | Balanced Accuracy: 0.5556
              precision    recall  f1-score   support

           0       0.55      0.63      0.59        27
           1       0.57      0.48      0.52        27

    accuracy                           0.56        54
   macro avg       0.56      0.56      0.55        54
weighted avg       0.56      0.56      0.55        54


--- Fold 2 ---
Supporto Test Set (Coorte Reale): {1: np.int64(27), 0: np.int64(27)}
Accuracy: 0.5741 | Balanced Accuracy: 0.5741
              precision    recall  f1-score   support

           0       0.60      0.44      0.51        27
           1       0.56      0.70      0.62        27

    accuracy                           0.57        54
   macro avg       0.58      0.57      0.57        54
weighted avg       0.58      0.57      0.57        54


--- Fold 3 ---
Supporto Test Set (Coorte

# Controllo il numero delle classi

In [53]:
for m in marker:
    vc_bal = y.value_counts()
    print("Negativi (classe 0):", vc_bal.get(0, 0))
    print("Positivi (classe 1):", vc_bal.get(1, 0))
    print(f"Totale pazienti usati: {len(y)}")


Negativi (classe 0): 123
Positivi (classe 1): 123
Totale pazienti usati: 246
Negativi (classe 0): 123
Positivi (classe 1): 123
Totale pazienti usati: 246
